# Fine-tuning with QLoRA (5-fold CV)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DSPagan/llms-time-complexity/blob/main/notebooks/fine_tuning.ipynb)

Fine-tune `Llama 3.1 8B Instruct` (4-bit) with QLoRA under **5-fold cross-validation**: for each fold, train on the other four and evaluate on the held-out one, reporting **mean ± std**. The logic lives in `src/`; this notebook wires it together.

In [ ]:
# Unsloth pulls its own compatible stack; Colab already provides a CUDA-enabled PyTorch.
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo

In [ ]:
# Clone the repo to get the code (src/) and the CodeComplex data.
!git clone https://github.com/DSPagan/llms-time-complexity.git
%cd llms-time-complexity

In [ ]:
import os, sys, json, gc
import numpy as np
import torch

sys.path.insert(0, os.getcwd())

from unsloth import FastLanguageModel
from src.load_model import load_model
from src.train_model import train_model
from src.prompts import build_prompt
from src.prepare_data import load_clean, stratified_folds, write_jsonl
from src.evaluate import evaluate, plot_confusion_matrix

In [ ]:
max_seq_length = 2048
num_epochs = 2
K = 5
folds = stratified_folds(load_clean(), k=K, seed=42)
print("fold sizes:", [len(f) for f in folds])

def predict(model, tokenizer, src, max_new_tokens=16):
    messages = [{"role": "user", "content": build_prompt(src)}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    if inputs.shape[1] > max_seq_length:
        return None
    out = model.generate(
        input_ids=inputs, do_sample=False, max_new_tokens=max_new_tokens,
        use_cache=True, no_repeat_ngram_size=4,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip()

## Cross-validation

Each fold reloads a fresh base model, fine-tunes on the other four folds, and evaluates on the held-out fold — so the runs are independent. This is the slow part: five full QLoRA trainings.

In [ ]:
accs, f1s, cm_sum = [], [], None
for i in range(K):
    print(f"\n===== Fold {i} =====")
    test = folds[i]
    train = [x for j in range(K) if j != i for x in folds[j]]
    write_jsonl(train, "fold_train.jsonl")

    model, tokenizer = load_model(max_seq_length=max_seq_length)          # fresh base each fold
    model, _ = train_model("fold_train.jsonl", model, tokenizer,
                           num_epochs=num_epochs, max_seq_length=max_seq_length)

    FastLanguageModel.for_inference(model)
    raw = [predict(model, tokenizer, item["src"]) for item in test]
    res = evaluate([item["complexity"] for item in test], raw)
    accs.append(res["accuracy"]); f1s.append(res["macro_f1"])
    cm_sum = res["confusion_matrix"] if cm_sum is None else cm_sum + res["confusion_matrix"]
    print(f"fold {i}: accuracy={res['accuracy']:.3f}  macro_f1={res['macro_f1']:.3f}")

    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()

## Results

In [ ]:
print(f"Fine-tuned (QLoRA, {num_epochs} epochs), {K}-fold CV:")
print(f"  accuracy  {np.mean(accs):.3f} +/- {np.std(accs, ddof=1):.3f}")
print(f"  macro_f1  {np.mean(f1s):.3f} +/- {np.std(f1s, ddof=1):.3f}")

os.makedirs("figures", exist_ok=True)
plot_confusion_matrix(cm_sum, title=f"Fine-tuned (QLoRA, {K}-fold CV)",
                      save_path="figures/CM_finetuned.png")